# Workshop-3 · Part A · EDA, cleaning and schema harmonization

**Course:** ETL (G01) · **Official guide:** Workshop-3, Part A, Step 1 (EDA) and Step 2 (cleaning and harmonization).

**Goal:** profile the five World Happiness files (2015-2019), document every data quality issue, harmonize the files into one unified schema and save the result as the input of the next step (feature engineering and model training).

| | |
|---|---|
| Input | `data/raw/2015.csv` ... `data/raw/2019.csv` |
| Output | `data/processed/happiness_unified.csv` |
| Deliverables of this notebook | EDA (sections 1-10), unified schema proposal (section 11), data quality observations (section 13) |

**Contents:** 0 Setup · 1 Load · 2 Structure · 3 Column names · 4 Data types · 5 Missing values · 6 Duplicated records · 7 Country names · 8 Target leakage · 9 Unified dataset · 10 Outliers and value ranges · 11 Unified schema proposal · 12 Save and final checks · 13 Data quality observations · 14 Summary of cleaning decisions

Descriptive statistics and visualizations of the relationships between variables (Step 3 of the guide) are in `model_training.ipynb`.

## 0. Setup
Paths are resolved from the project root, so the notebook runs from any folder inside the project. Run it with the project's `.venv` kernel.

In [3]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", None)

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2015, 2016, 2017, 2018, 2019]

RISKS = []   # data quality observations: filled along the notebook, shown in section 13


def add_risk(dataset, attribute, evidence, risk, requirement):
    RISKS.append({"Dataset / Attribute": f"{dataset} / {attribute}", "Evidence": evidence,
                  "Potential quality risk": risk, "Related requirement": requirement})


def pct(part, whole):
    return round(100 * part / whole, 2) if whole else 0.0


def structure(df):
    return pd.DataFrame({"dtype": df.dtypes.astype(str), "non_null": df.notna().sum(), "n_unique": df.nunique()})


def completeness(df):
    m = df.isna().sum()
    out = pd.DataFrame({"missing": m, "missing_pct": (100 * m / len(df)).round(2)})
    return out[out["missing"] > 0]


print("Project root:", PROJECT_ROOT)

Project root: c:\Users\Daniel\Desktop\UAO\V Semestre\ETL\ETL_2026-2_Workshop-3


## 1. Load the five raw files (read-only)

In [4]:
raw = {y: pd.read_csv(RAW_DIR / f"{y}.csv") for y in YEARS}

shapes = pd.DataFrame({y: {"rows": df.shape[0], "columns": df.shape[1]} for y, df in raw.items()}).T
shapes.index.name = "year"
display(shapes)
print(f"Total raw rows: {shapes['rows'].sum()}")

,rows,columns
year,,
2015,158,12
2016,157,13
2017,155,12
2018,156,9
2019,156,9


Total raw rows: 782


## 2. Structure: schema differences between years
Column names, data types and non-null counts of each file as they come from the source.

In [5]:
for y, df in raw.items():
    print(f"--- {y}: {df.shape[0]} rows x {df.shape[1]} columns")
    display(structure(df))

print("Files with a year column:", [y for y, df in raw.items() if any("year" in c.lower() for c in df.columns)] or "none")

--- 2015: 158 rows x 12 columns


,dtype,non_null,n_unique
Country,str,158,158
Region,str,158,10
Happiness Rank,int64,158,157
Happiness Score,float64,158,157
Standard Error,float64,158,153
Economy (GDP per Capita),float64,158,158
Family,float64,158,158
Health (Life Expectancy),float64,158,157
Freedom,float64,158,158
Trust (Government Corruption),float64,158,157


--- 2016: 157 rows x 13 columns


,dtype,non_null,n_unique
Country,str,157,157
Region,str,157,10
Happiness Rank,int64,157,154
Happiness Score,float64,157,154
Lower Confidence Interval,float64,157,154
Upper Confidence Interval,float64,157,154
Economy (GDP per Capita),float64,157,157
Family,float64,157,157
Health (Life Expectancy),float64,157,156
Freedom,float64,157,157


--- 2017: 155 rows x 12 columns


,dtype,non_null,n_unique
Country,str,155,155
Happiness.Rank,int64,155,155
Happiness.Score,float64,155,151
Whisker.high,float64,155,155
Whisker.low,float64,155,155
Economy..GDP.per.Capita.,float64,155,155
Family,float64,155,155
Health..Life.Expectancy.,float64,155,155
Freedom,float64,155,155
Generosity,float64,155,155


--- 2018: 156 rows x 9 columns


,dtype,non_null,n_unique
Overall rank,int64,156,156
Country or region,str,156,156
Score,float64,156,154
GDP per capita,float64,156,147
Social support,float64,156,146
Healthy life expectancy,float64,156,143
Freedom to make life choices,float64,156,136
Generosity,float64,156,122
Perceptions of corruption,float64,155,110


--- 2019: 156 rows x 9 columns


,dtype,non_null,n_unique
Overall rank,int64,156,156
Country or region,str,156,156
Score,float64,156,155
GDP per capita,float64,156,146
Social support,float64,156,145
Healthy life expectancy,float64,156,119
Freedom to make life choices,float64,156,130
Generosity,float64,156,118
Perceptions of corruption,float64,156,113


Files with a year column: none


**Observations**

- The number of rows and the number of columns change from file to file (see the table in section 1): the five files do not share one schema.
- No file has a `year` column. The year only exists in the file name, so it has to be added when the files are merged.
- Some columns exist only in some years (region, confidence intervals, whiskers, standard error, dystopia residual).
- The same indicator is written in three different ways across the files. Section 3 harmonizes them.

In [6]:
add_risk("All files", "year", "No file contains a year column; the year is only in the file name",
         "Stacking the files without adding the year loses the time dimension.", "PDF Step 2: resolve schema inconsistencies")
add_risk("All files", "number of columns", f"Columns per file: {dict(zip(YEARS, shapes['columns']))}",
         "Concatenating the files directly produces misaligned and sparse columns.", "PDF Step 1: schema differences between years")

## 3. Column names: cleaning and harmonization
The same indicator appears as `Economy (GDP per Capita)` (2015-2016), `Economy..GDP.per.Capita.` (2017) and `GDP per capita` (2018-2019). The column names are harmonized in two stages.

**Stage 1: mechanical cleaning (`clean_column_name`).** Everything is converted to lowercase, dots and parentheses are removed and any run of spaces or punctuation becomes a single underscore. Formatting differences disappear: `Economy (GDP per Capita)` and `Economy..GDP.per.Capita.` both become `economy_gdp_per_capita`.

**Stage 2: semantic renaming (`UNIFIED_NAMES`).** Cleaning is not enough for 2018-2019, where the same indicators use different words (`GDP per capita`, `Social support`, `Healthy life expectancy`...). Every cleaned name is mapped to one unified name: `country`, `score`, `gdp`, `family`, `health`, `freedom`, `generosity` and `corruption`.

In [7]:
def clean_column_name(name):
    # lowercase; dots, parentheses, spaces and any other punctuation -> one underscore; trim underscores
    return re.sub(r"[^0-9a-z]+", "_", name.strip().lower()).strip("_")


clean_raw = {}
name_rows = []
for y, df in raw.items():
    cleaned = df.copy()
    cleaned.columns = [clean_column_name(c) for c in df.columns]
    clean_raw[y] = cleaned
    for old, new in zip(df.columns, cleaned.columns):
        name_rows.append({"year": y, "raw_name": old, "clean_name": new})

name_log = pd.DataFrame(name_rows)
assert not name_log.duplicated(["year", "clean_name"]).any(), "two raw names collapsed into one inside a file"

print("Stage 1 result: raw name written in each year, grouped by cleaned name")
display(name_log.pivot(index="clean_name", columns="year", values="raw_name").fillna("-"))

Stage 1 result: raw name written in each year, grouped by cleaned name


year,2015,2016,2017,2018,2019
clean_name,,,,,
country,Country,Country,Country,-,-
country_or_region,-,-,-,Country or region,Country or region
dystopia_residual,Dystopia Residual,Dystopia Residual,Dystopia.Residual,-,-
economy_gdp_per_capita,Economy (GDP per Capita),Economy (GDP per Capita),Economy..GDP.per.Capita.,-,-
family,Family,Family,Family,-,-
freedom,Freedom,Freedom,Freedom,-,-
freedom_to_make_life_choices,-,-,-,Freedom to make life choices,Freedom to make life choices
gdp_per_capita,-,-,-,GDP per capita,GDP per capita
generosity,Generosity,Generosity,Generosity,Generosity,Generosity


After stage 1, the 2015-2017 files already agree on the names of the indicators, but 2018-2019 still use different words. Stage 2 maps every cleaned name to the unified name. Columns that cannot be used as model inputs or are not in all years are excluded, with the reason.

In [8]:
FEATURES = ["gdp", "family", "health", "freedom", "generosity", "corruption"]
UNIFIED_COLUMNS = ["country", "year"] + FEATURES + ["score"]

# stage 2: cleaned name -> unified name
UNIFIED_NAMES = {
    "country": "country", "country_or_region": "country",
    "happiness_score": "score", "score": "score",
    "economy_gdp_per_capita": "gdp", "gdp_per_capita": "gdp",
    "family": "family", "social_support": "family",
    "health_life_expectancy": "health", "healthy_life_expectancy": "health",
    "freedom": "freedom", "freedom_to_make_life_choices": "freedom",
    "trust_government_corruption": "corruption", "perceptions_of_corruption": "corruption",
    "generosity": "generosity",
}

# columns that are not kept, with the reason
EXCLUDED = {
    "region": "Only in 2015-2016 and not part of the event contract.",
    "happiness_rank": "Rank computed from the score: target leakage.",
    "overall_rank": "Rank computed from the score: target leakage.",
    "standard_error": "Uncertainty of the score estimate, derived from the target: target leakage.",
    "lower_confidence_interval": "Interval built around the score: target leakage.",
    "upper_confidence_interval": "Interval built around the score: target leakage.",
    "whisker_high": "Interval built around the score: target leakage.",
    "whisker_low": "Interval built around the score: target leakage.",
    "dystopia_residual": "Remainder of the score (score = six indicators + residual, verified in section 8): target leakage.",
}

for y, df in clean_raw.items():
    unknown = set(df.columns) - set(UNIFIED_NAMES) - set(EXCLUDED)
    assert not unknown, f"{y}: columns not covered by the mapping: {unknown}"

mapping_rows = [{"unified": UNIFIED_NAMES[c], "year": y, "cleaned_source": c}
                for y, df in clean_raw.items() for c in df.columns if c in UNIFIED_NAMES]
print("Stage 2 result: cleaned source name used for each unified column")
display(pd.DataFrame(mapping_rows).pivot(index="unified", columns="year", values="cleaned_source")
        .loc[["country", "score"] + FEATURES])

excluded_rows = [{"excluded_column": c, "present_in_years": ", ".join(str(y) for y, df in clean_raw.items() if c in df.columns),
                  "reason": reason} for c, reason in EXCLUDED.items()]
print("Columns not kept")
display(pd.DataFrame(excluded_rows).set_index("excluded_column"))

Stage 2 result: cleaned source name used for each unified column


year,2015,2016,2017,2018,2019
unified,,,,,
country,country,country,country,country_or_region,country_or_region
score,happiness_score,happiness_score,happiness_score,score,score
gdp,economy_gdp_per_capita,economy_gdp_per_capita,economy_gdp_per_capita,gdp_per_capita,gdp_per_capita
family,family,family,family,social_support,social_support
health,health_life_expectancy,health_life_expectancy,health_life_expectancy,healthy_life_expectancy,healthy_life_expectancy
freedom,freedom,freedom,freedom,freedom_to_make_life_choices,freedom_to_make_life_choices
generosity,generosity,generosity,generosity,generosity,generosity
corruption,trust_government_corruption,trust_government_corruption,trust_government_corruption,perceptions_of_corruption,perceptions_of_corruption


Columns not kept


,present_in_years,reason
excluded_column,,
region,"2015, 2016",Only in 2015-2016 and not part of the event contract.
happiness_rank,"2015, 2016, 2017",Rank computed from the score: target leakage.
overall_rank,"2018, 2019",Rank computed from the score: target leakage.
standard_error,2015,"Uncertainty of the score estimate, derived from the target: target leakage."
lower_confidence_interval,2016,Interval built around the score: target leakage.
upper_confidence_interval,2016,Interval built around the score: target leakage.
whisker_high,2017,Interval built around the score: target leakage.
whisker_low,2017,Interval built around the score: target leakage.
dystopia_residual,"2015, 2016, 2017","Remainder of the score (score = six indicators + residual, verified in section 8): target leakage."


**Assumptions to document**

- `Family` (2015-2017) and `Social support` (2018-2019) are treated as the same indicator under a new name. The files do not state it explicitly.
- `Trust (Government Corruption)` (2015-2017) and `Perceptions of corruption` (2018-2019) are treated as the same indicator.
- `Health (Life Expectancy)` and `Healthy life expectancy`, and `Freedom` and `Freedom to make life choices`, are treated as the same indicator.

In [9]:
def harmonize(df_clean, year):
    out = df_clean.rename(columns=UNIFIED_NAMES)
    out = out[["country"] + FEATURES + ["score"]].copy()
    out.insert(1, "year", year)
    out["country"] = out["country"].str.strip()
    return out


harm = {y: harmonize(df, y) for y, df in clean_raw.items()}
add_risk("All files", "column names", "The same indicator is written in three ways, e.g. 'Economy (GDP per Capita)', 'Economy..GDP.per.Capita.' and 'GDP per capita'",
         "Merging by name without harmonization duplicates the indicators in separate columns.",
         "PDF Step 1: inconsistent column names; PDF Step 2: standardize column names")
add_risk("All files", "family / corruption", "'Family' became 'Social support' and 'Trust (Government Corruption)' became 'Perceptions of corruption' in 2018",
         "Assumption: they measure the same indicator under a new name; a definition change would introduce a small inconsistency.",
         "PDF Step 2: resolve schema inconsistencies")

## 4. Data types

In [10]:
print("Data type of each unified column after reading the files with the default settings")
display(pd.DataFrame({y: df.dtypes.astype(str) for y, df in harm.items()}))

# What happens when the 2018 file is read without NA parsing
raw_2018_text = pd.read_csv(RAW_DIR / "2018.csv", keep_default_na=False)
corr_col = "Perceptions of corruption"
not_numeric = raw_2018_text[pd.to_numeric(raw_2018_text[corr_col], errors="coerce").isna()]
print(f"\n2018 '{corr_col}' read without NA parsing -> dtype {raw_2018_text[corr_col].dtype}")
display(not_numeric[["Country or region", corr_col]])

assert all(pd.api.types.is_numeric_dtype(df[c]) for df in harm.values() for c in FEATURES + ["score"])
print("All indicators and the score are numeric in the five harmonized frames.")

Data type of each unified column after reading the files with the default settings


,2015,2016,2017,2018,2019
country,str,str,str,str,str
year,int64,int64,int64,int64,int64
gdp,float64,float64,float64,float64,float64
family,float64,float64,float64,float64,float64
health,float64,float64,float64,float64,float64
freedom,float64,float64,float64,float64,float64
generosity,float64,float64,float64,float64,float64
corruption,float64,float64,float64,float64,float64
score,float64,float64,float64,float64,float64



2018 'Perceptions of corruption' read without NA parsing -> dtype str


,Country or region,Perceptions of corruption
19,United Arab Emirates,N/A


All indicators and the score are numeric in the five harmonized frames.


**Observations**

- By default pandas turns the text `N/A` of the 2018 file into a missing value, so the column ends up numeric. Read as plain text, that single token would turn the whole column into text.
- After harmonization the six indicators and the score are numeric (`float64`) in every year, and `year` is an integer.

In [11]:
add_risk("2018", "Perceptions of corruption", "One cell holds the text 'N/A' inside a numeric column (United Arab Emirates)",
         "If read as text, the whole column becomes a string column; pandas converts it to NaN by default.",
         "PDF Step 1: inconsistent data types")

## 5. Missing values
The five harmonized frames are stacked (country names still as written in the files) to profile them together.

In [12]:
stacked = pd.concat(harm.values(), ignore_index=True)
print(f"Stacked rows: {len(stacked)}")

comp = completeness(stacked)
display(comp)

missing_rows = stacked[stacked.isna().any(axis=1)]
display(missing_rows)
assert len(missing_rows) == 1, "the documented decision assumes exactly one incomplete row"

Stacked rows: 782


,missing,missing_pct
corruption,1,0.13


,country,year,gdp,family,health,freedom,generosity,corruption,score
489,United Arab Emirates,2018,2.096,0.776,0.67,0.284,0.186,NaN,6.774


**Decision: drop the incomplete row (United Arab Emirates, 2018).**

- Replacing `N/A` with `0` would tell the model that the corruption indicator is literally zero. Zeros do occur in the real data (section 10), so an artificial zero could not be told apart from a real one and would bias the model.
- It is 1 row out of 782 (0.13%), so dropping it loses almost no information.
- The event contract requires numeric values for every indicator, so a row with a missing indicator could not be streamed as a valid event.

In [13]:
n_missing = len(missing_rows)
add_risk("2018", "corruption (United Arab Emirates)", f"{n_missing} missing value out of {len(stacked)} rows ({pct(n_missing, len(stacked))}%)",
         "A missing indicator breaks the event contract; filling it with 0 would bias the model.",
         "PDF Step 1: missing values; PDF Step 2: remove or handle missing values (decision: drop the row)")

## 6. Duplicated records

In [14]:
print("Fully duplicated rows:", int(stacked.duplicated().sum()))
print("Repeated (country, year) pairs, names as written in the files:", int(stacked.duplicated(["country", "year"]).sum()))
display(stacked.groupby("year")["country"].agg(rows="size", distinct_countries="nunique"))

Fully duplicated rows: 0
Repeated (country, year) pairs, names as written in the files: 0


,rows,distinct_countries
year,,
2015,158,158
2016,157,157
2017,155,155
2018,156,156
2019,156,156


No duplicated rows and no country is repeated inside a year. This check is repeated after the country names are unified (section 9), because merging name variants could create duplicates.

## 7. Country names
`country` becomes the Kafka event key and feeds `dim_country`, so every country must have one single spelling in all years.

In [15]:
def country_series(df):
    return df["country"] if "country" in df.columns else df["country_or_region"]


whitespace = sum(int((country_series(df) != country_series(df).str.strip()).sum()) for df in clean_raw.values())
print("Country names with leading or trailing spaces in the raw files:", whitespace)

raw_distinct = stacked["country"].nunique()
print("Distinct country names as written in the files:", raw_distinct)

presence = stacked.assign(present=1).pivot_table(index="country", columns="year", values="present", aggfunc="max", fill_value=0)
not_in_all = presence[presence.sum(axis=1) < len(YEARS)]
print(f"\nNames that do not appear in all {len(YEARS)} years: {len(not_in_all)}")
display(pd.DataFrame(np.where(not_in_all > 0, "x", "-"), index=not_in_all.index, columns=not_in_all.columns))

Country names with leading or trailing spaces in the raw files: 0
Distinct country names as written in the files: 170

Names that do not appear in all 5 years: 29


year,2015,2016,2017,2018,2019
country,,,,,
Angola,x,x,x,x,-
Belize,-,x,x,x,-
Central African Republic,x,-,x,x,x
Comoros,x,x,-,-,x
Djibouti,x,-,-,-,-
Gambia,-,-,-,-,x
Hong Kong,x,x,-,x,x
"Hong Kong S.A.R., China",-,-,x,-,-
Laos,x,x,-,x,x


Comparing the names that are present only in some years reveals **six cases where the same country is written differently**. Each case is unified with the criterion "keep the most recent form". Only one case is an exception, explained in the table.

In [16]:
COUNTRY_FIXES = {
    "Hong Kong S.A.R., China": "Hong Kong",
    "Taiwan Province of China": "Taiwan",
    "North Cyprus": "Northern Cyprus",
    "Macedonia": "North Macedonia",
    "Trinidad & Tobago": "Trinidad and Tobago",
    "Somaliland region": "Somaliland Region",
}

RULE_APPLIED = {
    "Hong Kong S.A.R., China": "Most recent and most common form; the long form appears only in 2017.",
    "Taiwan Province of China": "Most recent and most common form; the long form appears only in 2017.",
    "North Cyprus": "Most recent form (2018-2019).",
    "Macedonia": "Most recent form: the country is North Macedonia since 2019.",
    "Trinidad & Tobago": "Exception to 'most recent': the '&' form is the latest (2018-2019) but it is an abbreviation; the full-word form is kept to avoid a special character in the Kafka key and in the database.",
    "Somaliland region": "Most recent form (2016); only the capital R of 'Region' differs.",
}


def years_of(name):
    return ", ".join(str(y) for y in sorted(stacked.loc[stacked["country"] == name, "year"].unique()))


assert set(COUNTRY_FIXES) <= set(stacked["country"]), "a documented variant is not present in the data"

cases = pd.DataFrame([{"case": i + 1, "variant (replaced)": v, "years of variant": years_of(v),
                       "kept name": k, "years of kept name": years_of(k), "criterion": RULE_APPLIED[v]}
                      for i, (v, k) in enumerate(COUNTRY_FIXES.items())]).set_index("case")
display(cases)

fixed_names = stacked["country"].replace(COUNTRY_FIXES)
before, after = stacked["country"].nunique(), fixed_names.nunique()
assert (before, after) == (170, 164), (before, after)
print(f"Distinct country names: {before} -> {after} after unifying the six cases")

presence_fixed = stacked.assign(country=fixed_names, present=1).pivot_table(
    index="country", columns="year", values="present", aggfunc="max", fill_value=0)
print("Names present in all five years: "
      f"{int((presence.sum(axis=1) == len(YEARS)).sum())} before -> {int((presence_fixed.sum(axis=1) == len(YEARS)).sum())} after")

,variant (replaced),years of variant,kept name,years of kept name,criterion
case,,,,,
1,"Hong Kong S.A.R., China",2017,Hong Kong,"2015, 2016, 2018, 2019",Most recent and most common form; the long form appears only in 2017.
2,Taiwan Province of China,2017,Taiwan,"2015, 2016, 2018, 2019",Most recent and most common form; the long form appears only in 2017.
3,North Cyprus,"2015, 2016, 2017",Northern Cyprus,"2018, 2019",Most recent form (2018-2019).
4,Macedonia,"2015, 2016, 2017, 2018",North Macedonia,2019,Most recent form: the country is North Macedonia since 2019.
5,Trinidad & Tobago,"2018, 2019",Trinidad and Tobago,"2015, 2016, 2017",Exception to 'most recent': the '&' form is the latest (2018-2019) but it is an abbreviation; the full-word form is kept to avoid a special character in the Kafka key and in the database.
6,Somaliland region,2015,Somaliland Region,2016,Most recent form (2016); only the capital R of 'Region' differs.


Distinct country names: 170 -> 164 after unifying the six cases
Names present in all five years: 141 before -> 146 after


In [17]:
add_risk("All files", "country", f"{before} distinct names; six countries are written in two ways (see the table in section 7)",
         "The same country would appear as two different keys: wrong partitioning in Kafka and duplicated members in dim_country.",
         "PDF Step 1: inconsistent values; PDF Step 2: standardize / resolve schema inconsistencies")

## 8. Target leakage candidates
The excluded columns of section 3 (ranks, standard error, confidence intervals, whiskers and dystopia residual) are all derived from the score. The dystopia residual deserves a numerical check: if the score can be rebuilt from the six indicators plus the residual, the residual contains the target itself.

In [18]:
rows = []
for y in [2015, 2016, 2017]:
    gap = (harm[y]["score"] - harm[y][FEATURES].sum(axis=1) - clean_raw[y]["dystopia_residual"]).abs()
    rows.append({"year": y, "max |score - (six indicators + dystopia_residual)|": round(gap.max(), 4)})
leak_check = pd.DataFrame(rows).set_index("year")
display(leak_check)
print("dystopia_residual present in 2018-2019:", {y: "dystopia_residual" in clean_raw[y].columns for y in [2018, 2019]})

,max |score - (six indicators + dystopia_residual)|
year,
2015,0.0005
2016,0.0005
2017,0.0005


dystopia_residual present in 2018-2019: {2018: False, 2019: False}


In 2015-2017 the score is reproduced, up to rounding, by the six indicators plus the residual. The residual is therefore excluded: feeding it to the model would make the prediction trivial.

The six indicators alone are legitimate inputs (they are in the event contract). However, they are the components of the score (the remaining part is the dystopia residual, which is excluded), so **a good fit is expected in the model and it is not a sign of leakage**.

In [19]:
add_risk("2015-2017", "dystopia_residual", f"score = six indicators + residual, maximum difference {leak_check.iloc[:, 0].max()}",
         "Using it as a feature gives the model the target: artificially perfect predictions that would not exist in the stream.",
         "PDF Step 3: avoid target leakage")
add_risk("2015-2019", "ranks, standard error, intervals, whiskers", "Columns computed from the score",
         "Target leakage; also absent in the event contract.", "PDF Step 3: avoid target leakage")
add_risk("2015-2016", "region", "Present only in 2015 and 2016",
         "A feature missing in three of five years cannot be supplied by the stream; it is not part of the event contract.",
         "PDF Step 2: resolve schema inconsistencies")

## 9. Build the unified dataset
Apply the two decisions taken so far: unify the six country names (section 7) and drop the incomplete row (section 5).

In [20]:
unified = stacked.copy()
unified["country"] = unified["country"].replace(COUNTRY_FIXES)

rows_before = len(unified)
dropped = unified[unified.isna().any(axis=1)]
unified = unified.dropna().reset_index(drop=True)

print(f"Rows: {rows_before} -> {len(unified)} (dropped {len(dropped)})")
print("Dropped row:")
display(dropped)

print("Repeated (country, year) pairs after unifying names:", int(unified.duplicated(["country", "year"]).sum()))
assert unified.duplicated(["country", "year"]).sum() == 0

in_all = unified.groupby("country")["year"].nunique()
print(f"Countries: {unified['country'].nunique()} | present in all five years: {int((in_all == len(YEARS)).sum())} | "
      f"present in fewer years: {int((in_all < len(YEARS)).sum())}")
display(unified.groupby("year").size().rename("rows").to_frame())

Rows: 782 -> 781 (dropped 1)
Dropped row:


,country,year,gdp,family,health,freedom,generosity,corruption,score
489,United Arab Emirates,2018,2.096,0.776,0.67,0.284,0.186,NaN,6.774


Repeated (country, year) pairs after unifying names: 0
Countries: 164 | present in all five years: 145 | present in fewer years: 19


,rows
year,
2015,158
2016,157
2017,155
2018,155
2019,156


In [21]:
add_risk("Unified", "panel balance", f"{int((in_all < len(YEARS)).sum())} countries do not appear in all five years",
         "Unbalanced panel: some countries have fewer observations. Not a problem for a simple regression, but it is documented.",
         "PDF Step 1: data quality observations")

## 10. Outliers and value ranges

In [22]:
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < low) | (s > high)).sum()), round(low, 3), round(high, 3)


rows = []
for c in FEATURES + ["score"]:
    n, low, high = iqr_outliers(unified[c])
    rows.append({"variable": c, "min": unified[c].min(), "max": unified[c].max(),
                 "iqr_lower_limit": low, "iqr_upper_limit": high, "outliers": n, "outliers_pct": pct(n, len(unified))})
outliers = pd.DataFrame(rows).set_index("variable")
display(outliers)

print("Minimum and maximum of each variable by year (same scale in all years?)")
display(unified.groupby("year")[FEATURES + ["score"]].agg(["min", "max"]).round(2))

zero_rows = unified[(unified[FEATURES] == 0).any(axis=1)]
print(f"Rows with a 0 in at least one indicator: {len(zero_rows)}")
display(zero_rows)

,min,max,iqr_lower_limit,iqr_upper_limit,outliers,outliers_pct
variable,,,,,,
gdp,0.000,1.870766,-0.338,2.177,0,0.00
family,0.000,1.644000,0.184,2.015,10,1.28
health,0.000,1.141000,-0.112,1.360,0,0.00
freedom,0.000,0.724000,-0.020,0.862,0,0.00
generosity,0.000,0.838075,-0.094,0.503,17,2.18
corruption,0.000,0.551910,-0.099,0.309,67,8.58
score,2.693,7.769000,2.000,8.692,0,0.00


Minimum and maximum of each variable by year (same scale in all years?)


gdp       family       health       freedom       generosity       corruption       score      
      min   max    min   max    min   max     min   max        min   max        min   max   min   max
year                                                                                                 
2015  0.0  1.69    0.0  1.40    0.0  1.03     0.0  0.67        0.0  0.80        0.0  0.55  2.84  7.59
2016  0.0  1.82    0.0  1.18    0.0  0.95     0.0  0.61        0.0  0.82        0.0  0.51  2.90  7.53
2017  0.0  1.87    0.0  1.61    0.0  0.95     0.0  0.66        0.0  0.84        0.0  0.46  2.69  7.54
2018  0.0  1.65    0.0  1.64    0.0  1.03     0.0  0.72        0.0  0.60        0.0  0.46  2.90  7.63
2019  0.0  1.68    0.0  1.62    0.0  1.14     0.0  0.63        0.0  0.57        0.0  0.45  2.85  7.77

Rows with a 0 in at least one indicator: 30


,country,year,gdp,family,health,freedom,generosity,corruption,score
73,Indonesia,2015,0.828270,1.087080,0.637930,0.466110,0.515350,0.000000,5.399
101,Greece,2015,1.154060,0.929330,0.882130,0.076990,0.000000,0.013970,4.857
111,Iraq,2015,0.985490,0.818890,0.602370,0.000000,0.179220,0.137880,4.677
119,Congo (Kinshasa),2015,0.000000,1.001200,0.098060,0.226050,0.248340,0.076250,4.517
122,Sierra Leone,2015,0.330240,0.955710,0.000000,0.408400,0.214880,0.087860,4.507
147,Central African Republic,2015,0.078500,0.000000,0.066990,0.488790,0.238350,0.082890,3.678
233,Somalia,2016,0.000000,0.336130,0.114660,0.567780,0.272250,0.311800,5.440
244,Bosnia and Herzegovina,2016,0.933830,0.643670,0.707660,0.095110,0.298890,0.000000,5.163
256,Greece,2016,1.248860,0.754730,0.800290,0.058220,0.000000,0.041270,5.033
268,Sierra Leone,2016,0.364850,0.628000,0.000000,0.306850,0.238970,0.081960,4.635


**Decisions**

- **Outliers are kept.** The values flagged by the 1.5 x IQR rule are valid observations inside the bounded range of each indicator (the distributions are skewed), not measurement errors. Removing them would delete real countries.
- **Zeros are kept.** They look like real minimum values of the indicators, not missing data: the same countries repeat across years (for example Greece has `generosity` = 0 in all five years and the Central African Republic has `family` = 0 in several). This is also why replacing a missing value with 0 would have been wrong (section 5).
- **No rescaling between years.** The six indicators share the same range in all years. Any scaling needed by the model belongs to the model pipeline, not to this cleaning step.

In [23]:
flagged = outliers[outliers["outliers"] > 0]
add_risk("Unified", "outliers (1.5 x IQR)", "; ".join(f"{i}: {int(r.outliers)} ({r.outliers_pct}%)" for i, r in flagged.iterrows()) or "none",
         "Extreme values could distort a linear model; they are valid observations, so they are kept and documented.",
         "PDF Step 1: potential outliers")
add_risk("Unified", "zeros in indicators", f"{len(zero_rows)} rows with a 0 in at least one indicator",
         "Zeros could be mistaken for missing data; they look like real minimum values. Kept and documented.",
         "PDF Step 1: potential outliers")

## 11. Unified schema proposal

| Unified column | Type | Meaning | Source 2015-2016 | Source 2017 | Source 2018-2019 | Event field |
|---|---|---|---|---|---|---|
| `country` | string | Country name, one spelling per country (164 distinct) | Country | Country | Country or region | `country` (also the Kafka key) |
| `year` | integer | Observation year, taken from the file name | (file name) | (file name) | (file name) | `year` |
| `gdp` | float | GDP per capita contribution | Economy (GDP per Capita) | Economy..GDP.per.Capita. | GDP per capita | `gdp` |
| `family` | float | Family / social support contribution | Family | Family | Social support | `family` |
| `health` | float | Healthy life expectancy contribution | Health (Life Expectancy) | Health..Life.Expectancy. | Healthy life expectancy | `health` |
| `freedom` | float | Freedom to make life choices contribution | Freedom | Freedom | Freedom to make life choices | `freedom` |
| `generosity` | float | Generosity contribution | Generosity | Generosity | Generosity | `generosity` |
| `corruption` | float | Perception of corruption contribution | Trust (Government Corruption) | Trust..Government.Corruption. | Perceptions of corruption | `corruption` |
| `score` | float | Happiness score (target) | Happiness Score | Happiness.Score | Score | `actual_happiness_score` (evaluation only, never a model input) |

`event_id` and `event_time` are not part of the dataset: the Kafka producer generates them for every event.

The unified names already match the event fields, so the producer only has to rename `score` to `actual_happiness_score` and add `event_id` and `event_time`.

In [24]:
display(structure(unified))

,dtype,non_null,n_unique
country,str,781,164
year,int64,781,5
gdp,float64,781,741
family,float64,781,731
health,float64,781,704
freedom,float64,781,696
generosity,float64,781,663
corruption,float64,781,635
score,float64,781,715


## 12. Save the processed dataset and run the final checks

In [25]:
OUTPUT_PATH = PROCESSED_DIR / "happiness_unified.csv"
unified[UNIFIED_COLUMNS].to_csv(OUTPUT_PATH, index=False)

check = pd.read_csv(OUTPUT_PATH)
assert list(check.columns) == UNIFIED_COLUMNS
assert check.isna().sum().sum() == 0
assert not check.duplicated(["country", "year"]).any()
assert check["country"].nunique() == 164
assert len(check) == 781

display(check.groupby("year").size().rename("rows").to_frame())
print(f"Saved {len(check)} rows x {check.shape[1]} columns to {OUTPUT_PATH.relative_to(PROJECT_ROOT)}")
print("All final checks passed.")

,rows
year,
2015,158
2016,157
2017,155
2018,155
2019,156


Saved 781 rows x 9 columns to data\processed\happiness_unified.csv
All final checks passed.


## 13. Data quality observations
Register filled along the notebook: evidence found, potential risk and the requirement of the guide it relates to.

In [26]:
risk_register = pd.DataFrame(RISKS)
risk_register.index = risk_register.index + 1
risk_register.index.name = "#"
display(risk_register)
print(f"{len(risk_register)} observations registered.")

,Dataset / Attribute,Evidence,Potential quality risk,Related requirement
#,,,,
1,All files / year,No file contains a year column; the year is only in the file name,Stacking the files without adding the year loses the time dimension.,PDF Step 2: resolve schema inconsistencies
2,All files / number of columns,"Columns per file: {2015: 12, 2016: 13, 2017: 12, 2018: 9, 2019: 9}",Concatenating the files directly produces misaligned and sparse columns.,PDF Step 1: schema differences between years
3,All files / column names,"The same indicator is written in three ways, e.g. 'Economy (GDP per Capita)', 'Economy..GDP.per.Capita.' and 'GDP per capita'",Merging by name without harmonization duplicates the indicators in separate columns.,PDF Step 1: inconsistent column names; PDF Step 2: standardize column names
4,All files / family / corruption,'Family' became 'Social support' and 'Trust (Government Corruption)' became 'Perceptions of corruption' in 2018,Assumption: they measure the same indicator under a new name; a definition change would introduce a small inconsistency.,PDF Step 2: resolve schema inconsistencies
5,2018 / Perceptions of corruption,One cell holds the text 'N/A' inside a numeric column (United Arab Emirates),"If read as text, the whole column becomes a string column; pandas converts it to NaN by default.",PDF Step 1: inconsistent data types
6,2018 / corruption (United Arab Emirates),1 missing value out of 782 rows (0.13%),A missing indicator breaks the event contract; filling it with 0 would bias the model.,PDF Step 1: missing values; PDF Step 2: remove or handle missing values (decision: drop the row)
7,All files / country,170 distinct names; six countries are written in two ways (see the table in section 7),The same country would appear as two different keys: wrong partitioning in Kafka and duplicated members in dim_country.,PDF Step 1: inconsistent values; PDF Step 2: standardize / resolve schema inconsistencies
8,2015-2017 / dystopia_residual,"score = six indicators + residual, maximum difference 0.0005",Using it as a feature gives the model the target: artificially perfect predictions that would not exist in the stream.,PDF Step 3: avoid target leakage
9,"2015-2019 / ranks, standard error, intervals, whiskers",Columns computed from the score,Target leakage; also absent in the event contract.,PDF Step 3: avoid target leakage


13 observations registered.


## 14. Summary of cleaning decisions

| # | Decision | Justification |
|---|---|---|
| 1 | Clean column names in two stages (lowercase and remove dots/parentheses, then map to unified names) | The same indicator is written in three ways; harmonization is required before merging |
| 2 | Add `year` from the file name | No file has a year column; it is required by the event contract |
| 3 | Exclude ranks, standard error, intervals, whiskers and dystopia residual | They are derived from the score: target leakage |
| 4 | Exclude `region` | Present only in 2015-2016 and not part of the event contract |
| 5 | Unify six country names (170 -> 164 distinct names) | `country` is the Kafka key and feeds `dim_country`: one spelling per country |
| 6 | Drop the United Arab Emirates 2018 row (missing corruption) | 1 of 782 rows; filling it with 0 would bias the model and the event contract needs numeric values |
| 7 | Keep outliers and zeros | Valid observations inside the bounded range of each indicator |
| 8 | No rescaling between years | The indicators share the same range in all years |
| 9 | Save `data/processed/happiness_unified.csv` (781 rows, 9 columns) | Single input for the model training and for the Kafka producer |

**Next step:** `model_training.ipynb` (descriptive statistics and visualizations, feature selection with justification, scikit-learn `Pipeline`, 70/30 split, MAE, RMSE and R², `model.pkl`).